In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
Using cached sympy-1.14.0-py3-none-any.whl

In [9]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit.result import marginal_counts
from qiskit.circuit.library import UnitaryGate
import math 

# Useful constants
root2 = math.sqrt(2)
denom1 = math.sqrt(4+2*root2)
denom2 = math.sqrt(4-2*root2)

W_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]
                    
W_transform = Operator(W_transform_matrix) 

V_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]
V_transform = Operator(V_transform_matrix)

# Construct a circuit that prepares the state  1/sqrt(2)( |01> - |10> )
def entangledPair():
    q = QuantumCircuit(2) 
    q.h(0)
    q.cx(0, 1)
    q.x(0)
    q.z(1)
    return q

# Generates a random bit with P(1)=2/3.
def get_random_basis():
    matrix = [
        [math.sqrt(1/3), -math.sqrt(2/3)],
        [math.sqrt(2/3),  math.sqrt(1/3)]
    ]
    custom_gate = UnitaryGate(matrix, label="Prob1/3")
    
    qc = QuantumCircuit(1)
    qc.append(custom_gate, [0])
    qc.measure_all()
    
    backend = BasicSimulator()
    job = backend.run(qc, shots=1)
    result = job.result()
    
    outcome = int(list(result.get_counts().keys())[0])
    return outcome

# Choose the base for each person randomly
def choix_base(name):
    baseAlice = ["X","W","Z"]
    baseBob = ["W","Z","V"]
    choice = get_random_basis()
    choice_nb = 0
    if(choice==0):
        choice_nb=0 #choice base index 0
    else:
        choice = get_random_basis()
        if choice == 0:
            choice_nb=1 #choice base index 1
        else:
            choice_nb=2 #choice base index 2
        
    if(name == "alice"):
        return baseAlice[choice_nb]
    elif(name=="bob"):
        return baseBob[choice_nb]

# Apply basis rotation to the qubit.
def application_base(value, base, bit):
    match base:
        case "X":
            value.h(bit)
        case "V":
            value.unitary(V_transform_matrix,[bit])
        case "W":
            value.unitary(W_transform_matrix,[bit])
    return value
        
# Initialisation of variables
N = 200
key=[]
basesAlice = []
basesBob = []
resultsAlice = []
resultsBob=[]

nXxV=0;XxV=0
nXxW=0;XxW=0
nZxV=0;ZxV=0
nZxW=0;ZxW=0
 
for i in range ((9*N)//2):
    # We create a new entrangled pair
    entrangled_pair=entangledPair()

    # We chose randomly the base for the measurement 
    choice_baseAlice = choix_base("alice")
    choice_baseBob = choix_base("bob")

    # We apply the base to the entrangled pair
    circuit=application_base(entrangled_pair,choice_baseAlice,0)  
    circuit=application_base(circuit,choice_baseBob,1)

    # Measure both qubits and store the bases
    circuit.measure_all()
    basesAlice.append(choice_baseAlice)
    basesBob.append(choice_baseBob)
    backend = BasicSimulator()
    compiled = transpile(circuit, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts=result_sim.get_counts(compiled)

    # Extract the measurement results
    bitstring=list(counts.keys())[0]
    bit_alice = int(bitstring[1])
    bit_bob = int(bitstring[0])
    
    resultsAlice.append(bit_alice)
    resultsBob.append(bit_bob)

for i in range ((9*N)//2):
        # If bases are the same we add the value of the qbit in the key
        if(basesAlice[i]==basesBob[i]):
            # Check for perfect anti-correlation (singlet state)
            # This means that an attacker try to intercept theses qbits
            if(resultsAlice[i]!=resultsBob[i]):
                key.append(resultsAlice[i])
            else:
                print("Intrication error")
        else:
            # We convert the 0 and 1 to 1 and -1
            resultsAlice_conv=(-1)**resultsAlice[i]
            resultsBob_conv=(-1)**resultsBob[i]
            # We calculate the average values for the differents combinaison of base
            if(basesAlice[i]=="X"):
                if(basesBob[i]=="W"):
                    XxW+=resultsAlice_conv*resultsBob_conv
                    nXxW+=1
                elif(basesBob[i]=="V"):
                    XxV+=resultsAlice_conv*resultsBob_conv
                    nXxV+=1
            elif(basesAlice[i]=="Z"):
                if(basesBob[i]=="W"):
                    ZxW+=resultsAlice_conv*resultsBob_conv
                    nZxW+=1
                elif(basesBob[i]=="V"):
                    ZxV+=resultsAlice_conv*resultsBob_conv
                    nZxV+=1

# Calculate average values
XxW_avg = (XxW/nXxW) if nXxW > 0 else 0
XxV_avg = (XxV/nXxV) if nXxV > 0 else 0
ZxW_avg = (ZxW/nZxW) if nZxW > 0 else 0
ZxV_avg = (ZxV/nZxV) if nZxV > 0 else 0
S = abs(XxW_avg-XxV_avg+ZxW_avg+ZxV_avg)
print(S)
print(key)

2.8462514189738117
[0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1]
